# 🌐 WikiGame Bot
Navigate from any Wikipedia page to any target using AI (GLiNER2 + embeddings + GraphRAG + Llama).

**Make sure this notebook is inside your `files/` folder** (same level as `bot.py`).

In [ ]:
# ── Cell 1: Setup — run this first, once ─────────────────────────────────────
import sys, os

# Make sure we can import core/, graph/, etc.
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

print(f'✅ Project root: {NOTEBOOK_DIR}')
print(f'Python: {sys.version.split()[0]}')

# Verify all modules are findable
missing = []
for mod in ['core', 'graph', 'search', 'llm']:
    if not os.path.isdir(os.path.join(NOTEBOOK_DIR, mod)):
        missing.append(mod)

if missing:
    print(f'❌ Missing folders: {missing}')
    print('   Make sure this notebook is in the same folder as bot.py')
else:
    print('✅ All module folders found')

In [ ]:
# ── Cell 2: Install / verify packages ────────────────────────────────────────
# Run this if you haven't installed requirements yet, or want to verify
import subprocess

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✅ All packages installed')
else:
    print('⚠️  Some packages may have issues:')
    print(result.stderr[-500:])

In [ ]:
# ── Cell 3: Load all components ───────────────────────────────────────────────
# This takes ~30 seconds the first time (downloads embedding model)
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import time

print('Loading components...')

import config
from core.wiki_fetcher import WikiFetcher
from core.embedder import Embedder
from graph.graph_rag import GraphRAG
from graph.wild_graph import WildGraph
from search.gliner_filter import GlinerFilter
from search.ranker import Ranker
from llm.llama_agent import LlamaAgent

print('  ✅ WikiFetcher')

embedder = Embedder()
print('  ✅ Embedder (sentence-transformers)')

gliner = GlinerFilter()
print('  ✅ GLiNER2')

llama = LlamaAgent()
if llama.is_available():
    print(f'  ✅ Ollama ({config.OLLAMA_MODEL})')
else:
    print(f'  ⚠️  Ollama not available — will use embedding-only mode')
    llama = None

print('\n🚀 Ready to play!')

In [ ]:
# ── Cell 4: Rich display helpers ─────────────────────────────────────────────

def render_hop(hop_num, from_page, to_page, score, reasoning='', candidates=None):
    """Render a single hop as a styled HTML card."""
    cand_html = ''
    if candidates:
        bars = ''
        max_score = max(s for _, s in candidates[:6]) or 1
        for title, s in candidates[:6]:
            pct = s / max_score * 100
            is_chosen = title == to_page
            color = '#48bb78' if is_chosen else '#7c6af7'
            bold = 'font-weight:700;' if is_chosen else ''
            check = ' ✓' if is_chosen else ''
            bars += f'''
            <div style="display:flex;align-items:center;gap:8px;margin:4px 0">
              <div style="font-size:11px;color:#a0aec0;min-width:160px;{bold}white-space:nowrap;overflow:hidden;text-overflow:ellipsis">{title}{check}</div>
              <div style="flex:1;height:4px;background:#2d3748;border-radius:2px">
                <div style="width:{pct:.0f}%;height:100%;background:{color};border-radius:2px"></div>
              </div>
              <div style="font-size:10px;color:#718096;font-family:monospace;min-width:36px">{s:.3f}</div>
            </div>'''
        cand_html = f'<div style="margin-top:10px;padding-top:10px;border-top:1px solid #2d3748">{bars}</div>'

    reasoning_html = f'<div style="font-size:11px;color:#718096;font-style:italic;margin-top:6px">💭 {reasoning}</div>' if reasoning else ''

    return f'''
    <div style="font-family:-apple-system,sans-serif;background:#1a1a2e;border:1px solid #2d3748;
                border-radius:12px;padding:14px 18px;margin:6px 0;animation:fadeIn 0.3s">
      <div style="display:flex;align-items:center;gap:10px">
        <div style="background:#7c6af7;color:white;border-radius:6px;padding:3px 8px;
                    font-size:11px;font-weight:700;font-family:monospace">HOP {hop_num}</div>
        <div style="color:#718096;font-size:12px">{from_page}</div>
        <div style="color:#4fd1c5;font-size:16px">→</div>
        <div style="color:#e2e8f0;font-size:14px;font-weight:600">{to_page}</div>
        <div style="margin-left:auto;font-family:monospace;font-size:12px;color:#f6ad55">{score:.3f}</div>
      </div>
      {reasoning_html}
      {cand_html}
    </div>'''

def render_header(start, target):
    return f'''
    <div style="font-family:-apple-system,sans-serif;background:#0f0f1a;border:1px solid #7c6af7;
                border-radius:14px;padding:20px 24px;margin-bottom:12px">
      <div style="font-size:20px;font-weight:700;color:white;margin-bottom:10px">🌐 WikiGame Bot</div>
      <div style="display:flex;gap:24px">
        <div><div style="font-size:10px;color:#718096;text-transform:uppercase;letter-spacing:1px">Start</div>
             <div style="font-size:15px;color:#4fd1c5;font-weight:600">{start}</div></div>
        <div style="color:#4a5568;font-size:20px;align-self:center">→</div>
        <div><div style="font-size:10px;color:#718096;text-transform:uppercase;letter-spacing:1px">Target</div>
             <div style="font-size:15px;color:#7c6af7;font-weight:600">{target}</div></div>
      </div>
    </div>'''

def render_result(success, path, hops, elapsed):
    if success:
        path_str = ' → '.join(path)
        return f'''
        <div style="font-family:-apple-system,sans-serif;background:#0f2a1a;border:2px solid #48bb78;
                    border-radius:14px;padding:20px 24px;margin-top:12px;text-align:center">
          <div style="font-size:32px;margin-bottom:8px">🏆</div>
          <div style="font-size:18px;font-weight:700;color:#48bb78">Target reached in {hops} hops!</div>
          <div style="font-size:12px;color:#718096;margin-top:4px">Time: {elapsed:.1f}s</div>
          <div style="background:#1a2d1a;border-radius:8px;padding:12px;margin-top:14px;
                      font-family:monospace;font-size:12px;color:#68d391;text-align:left;word-break:break-word">
            {path_str}
          </div>
        </div>'''
    else:
        return f'''
        <div style="font-family:-apple-system,sans-serif;background:#2a0f0f;border:2px solid #fc8181;
                    border-radius:14px;padding:20px 24px;margin-top:12px;text-align:center">
          <div style="font-size:18px;font-weight:700;color:#fc8181">❌ Failed after {hops} hops</div>
          <div style="font-size:12px;color:#718096;margin-top:4px">Try increasing MAX_HOPS in config.py or switching to beam search</div>
        </div>'''

print('✅ Display helpers loaded')

In [ ]:
# ── Cell 5: The main play() function ─────────────────────────────────────────
import numpy as np

def play(
    start: str,
    target: str,
    strategy: str = 'beam',       # 'greedy' or 'beam'
    beam_width: int = 3,
    max_hops: int = 30,
    use_llm: bool = True,
    show_candidates: bool = True,  # show score bars for each hop
    verbose: bool = False,
):
    """
    Play the Wikipedia game from start → target.
    Displays rich hop-by-hop output inline in the notebook.
    """
    fetcher = WikiFetcher()
    graph_rag = GraphRAG()
    wild_graph = WildGraph(strategy=strategy, beam_width=beam_width, max_hops=max_hops)
    ranker = Ranker(
        embedder=embedder,
        gliner_filter=gliner,
        graph_rag=graph_rag,
        llama_agent=llama if use_llm else None,
    )

    output = widgets.Output()
    display(output)
    start_time = time.time()

    with output:
        # ── Fetch pages ───────────────────────────────────────────────────────
        display(HTML('<div style="color:#a0aec0;font-family:monospace;font-size:13px">Fetching pages…</div>'))

        start_page = fetcher.fetch_page(start)
        target_page = fetcher.fetch_page(target)

        if not start_page:
            display(HTML(f'<div style="color:#fc8181">❌ Could not find start page: "{start}"</div>'))
            suggestions = fetcher.search(start, limit=3)
            if suggestions:
                display(HTML(f'<div style="color:#a0aec0">Did you mean: {", ".join(suggestions)}?</div>'))
            return

        if not target_page:
            display(HTML(f'<div style="color:#fc8181">❌ Could not find target page: "{target}"</div>'))
            suggestions = fetcher.search(target, limit=3)
            if suggestions:
                display(HTML(f'<div style="color:#a0aec0">Did you mean: {", ".join(suggestions)}?</div>'))
            return

        start_title = start_page.title
        target_title = target_page.title

        clear_output(wait=True)
        display(HTML(render_header(start_title, target_title)))

        if start_title.lower() == target_title.lower():
            display(HTML('<div style="color:#48bb78;font-size:16px">✅ Already there!</div>'))
            return

        # ── Set up ranker ─────────────────────────────────────────────────────
        ranker.set_target(target_title, target_page.lede)

        target_emb = embedder.embed(embedder.build_target_text(target_title, target_page.lede))
        graph_rag.add_node(target_title, embedding=target_emb,
                           lede=target_page.lede, categories=target_page.categories)
        graph_rag.add_page_links(target_title, target_page.links)

        wild_graph.initialize(start_title, target_title)
        visited = {start_title}

        start_emb = embedder.embed(embedder.build_target_text(start_title, start_page.lede))
        graph_rag.add_node(start_title, embedding=start_emb,
                           lede=start_page.lede, categories=start_page.categories)
        graph_rag.add_page_links(start_title, start_page.links)

        page_cache = {start_title: start_page, target_title: target_page}
        hop_number = 0
        all_hops_html = ''

        # ── Main loop ─────────────────────────────────────────────────────────
        while not wild_graph.is_done():
            hop_number += 1
            frontier = wild_graph.get_frontier()
            if not frontier:
                break

            for current_title in frontier:
                if current_title not in page_cache:
                    current_page = fetcher.fetch_page(current_title)
                    if not current_page:
                        continue
                    page_cache[current_title] = current_page
                else:
                    current_page = page_cache[current_title]

                cur_emb = embedder.embed(
                    embedder.build_target_text(current_title, current_page.lede))
                graph_rag.add_node(current_title, embedding=cur_emb,
                                   lede=current_page.lede, categories=current_page.categories)
                graph_rag.add_page_links(current_title, current_page.links)
                graph_rag.record_visit(current_title)
                visited.add(current_title)

                if not current_page.links:
                    continue

                ranked = ranker.rank(
                    current_page_title=current_title,
                    current_page_lede=current_page.lede,
                    candidate_links=current_page.links,
                    visited_pages=visited,
                    verbose=verbose,
                )

                if not ranked:
                    continue

                best_title, best_score, reasoning = ranked[0]
                candidates_for_display = [(t, s) for t, s, _ in ranked[:6]] if show_candidates else None

                # Render this hop
                hop_html = render_hop(
                    hop_num=hop_number,
                    from_page=current_title,
                    to_page=best_title,
                    score=best_score,
                    reasoning=reasoning,
                    candidates=candidates_for_display,
                )
                all_hops_html += hop_html

                # Redraw all hops so far
                clear_output(wait=True)
                display(HTML(render_header(start_title, target_title)))
                display(HTML(all_hops_html))

                wild_graph.advance(
                    from_page=current_title,
                    ranked_candidates=[(t, s) for t, s, _ in ranked],
                    reasoning=reasoning,
                )

                if wild_graph.is_done():
                    break

        # ── Final result ──────────────────────────────────────────────────────
        elapsed = time.time() - start_time
        result = wild_graph.get_result()

        clear_output(wait=True)
        display(HTML(render_header(start_title, target_title)))
        display(HTML(all_hops_html))
        display(HTML(render_result(
            success=result.success,
            path=result.path.pages if result.path else [],
            hops=result.hops_taken,
            elapsed=elapsed,
        )))

        return result

print('✅ play() function ready')

In [ ]:
# ── Cell 6: ▶ RUN A GAME — edit start and target here ────────────────────────

result = play(
    start  = "Pizza",
    target = "Alan Turing",

    # Options (all optional):
    strategy       = "beam",   # 'beam' (smarter) or 'greedy' (faster)
    beam_width     = 3,        # how many parallel paths to explore
    max_hops       = 30,       # give up after this many hops
    use_llm        = True,     # set False to skip Llama (faster)
    show_candidates= True,     # show score bars for each hop
)

In [ ]:
# ── Cell 7: Interactive widget UI ────────────────────────────────────────────
# Run this cell for a point-and-click interface instead of editing code

start_box  = widgets.Text(value='Pizza',      description='Start:', layout=widgets.Layout(width='300px'))
target_box = widgets.Text(value='Alan Turing',description='Target:', layout=widgets.Layout(width='300px'))
strategy_w = widgets.Dropdown(options=['beam','greedy'], value='beam', description='Strategy:')
beam_w     = widgets.IntSlider(value=3, min=1, max=5, description='Beam width:')
hops_w     = widgets.IntSlider(value=30, min=5, max=60, description='Max hops:')
llm_w      = widgets.Checkbox(value=True, description='Use Llama (Ollama)')
run_btn    = widgets.Button(description='▶  Run Game', button_style='success',
                            layout=widgets.Layout(width='160px', height='36px'))

def on_run(b):
    play(
        start=start_box.value,
        target=target_box.value,
        strategy=strategy_w.value,
        beam_width=beam_w.value,
        max_hops=hops_w.value,
        use_llm=llm_w.value,
    )

run_btn.on_click(on_run)

display(widgets.VBox([
    widgets.HBox([start_box, target_box]),
    widgets.HBox([strategy_w, beam_w]),
    widgets.HBox([hops_w, llm_w]),
    run_btn,
]))

In [ ]:
# ── Cell 8: Inspect last result ───────────────────────────────────────────────
# Run after a game to see full hop log

if result and result.path:
    print(f'Path ({result.hops_taken} hops):')
    for i, page in enumerate(result.path.pages):
        prefix = '📖' if i == 0 else ('🏆' if i == len(result.path.pages)-1 else f'  {i}.')
        print(f'  {prefix} {page}')

    print(f'\nHop scores: {[round(s,3) for s in result.path.scores]}')
    print(f'Average score: {result.path.average_score():.3f}')

In [ ]:
# ── Cell 9: Debug a specific page ────────────────────────────────────────────
# Useful for understanding why the bot chose a particular hop

DEBUG_PAGE  = "Italy"
DEBUG_TARGET = "Alan Turing"

fetcher = WikiFetcher()
page = fetcher.fetch_page(DEBUG_PAGE)
tgt  = fetcher.fetch_page(DEBUG_TARGET)

print(f'Page: {page.title}')
print(f'Lede: {page.lede[:200]}...')
print(f'Total links: {len(page.links)}')

# Score all links
from search.ranker import Ranker
from graph.graph_rag import GraphRAG
test_ranker = Ranker(embedder=embedder, gliner_filter=gliner,
                     graph_rag=GraphRAG(), llama_agent=None)
test_ranker.set_target(tgt.title, tgt.lede)

ranked = test_ranker.rank(page.title, page.lede, page.links, visited_pages=set())

print(f'\nTop 15 candidates toward "{DEBUG_TARGET}":')
for i, (title, score, _) in enumerate(ranked[:15], 1):
    bar = '█' * int(score * 30)
    print(f'  {i:2}. {title:45s} {score:.3f} {bar}')